#### Configuração do Contexto

In [ ]:
# Definindo o contexto de execução
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA synthea_data")

# Listando para garantir que o Spark está 'enxergando' as tabelas
display(spark.sql("SHOW TABLES"))

#### Leitura dos Dados (Lazy Evaluation)

In [ ]:
# Lendo as tabelas da camada Bronze
df_patients = spark.read.table("patients")
df_encounters = spark.read.table("encounters")
df_conditions = spark.read.table("conditions")

# Visualizando a estrutura (Schema) inferida
df_patients.printSchema()

#### Tratamento de Dados - Criando a Camada Silver

In [ ]:
from pyspark.sql.functions import col, to_date, datediff, current_date, floor, lit

# Transformação na Tabela de Pacientes
df_patients_silver = df_patients.select(
    col("Id").alias("patient_id"),
    # Tratando datas (Synthea geralmente traz datas no formato yyyy-MM-dd)
    to_date(col("BIRTHDATE")).alias("data_nascimento"),
    col("CITY").alias("cidade"),
    col("STATE").alias("estado"),
    col("GENDER").alias("genero"),
    col("RACE").alias("raca")
).withColumn(
    # Calculando Idade: (Hoje - Nascimento) / 365
    "idade", 
    floor(datediff(current_date(), col("data_nascimento")) / 365.25)
)

display(df_patients_silver.limit(5))

#### Joins e Cruzamento de Dados

In [ ]:
# 1. Preparando a tabela de Encontros (Focando em custos)
# Filtrar apenas encontros que tiveram custo
df_encounters_clean = df_encounters.select(
    col("Id").alias("encounter_id"),
    col("PATIENT").alias("patient_id"),
    col("TOTAL_CLAIM_COST").cast("double").alias("custo_total"),
    col("PAYER_COVERAGE").cast("double").alias("cobertura_convenio"),
    col("DESCRIPTION").alias("tipo_encontro"),
    col("START")
).fillna(0, subset=["custo_total", "cobertura_convenio"]) # Tratando nulos

# 2. Preparando a tabela de Condições (Diagnósticos)
df_conditions_clean = df_conditions.select(
    col("PATIENT").alias("patient_id"),
    col("ENCOUNTER").alias("encounter_id"),
    col("DESCRIPTION").alias("diagnostico")
)

# 3. Realizando o JOIN (Silver Consolidada)
df_full_silver = df_encounters_clean.join(
    df_patients_silver, 
    on="patient_id", 
    how="inner"
).join(
    df_conditions_clean,
    on=["patient_id", "encounter_id"], # Join em chaves compostas é mais seguro
    how="left" # Left join, pois nem todo encontro gera um diagnóstico de doença crônica
)

display(df_full_silver.limit(5))

### Otimizando com broadcast

In [ ]:
# 1. Preparando a tabela de Encontros (Renomeando para o teste otimizado)
df_encounters_opt = df_encounters.select(
    col("Id").alias("encounter_id"),
    col("PATIENT").alias("patient_id"),
    col("TOTAL_CLAIM_COST").cast("double").alias("custo_total"),
    col("PAYER_COVERAGE").cast("double").alias("cobertura_convenio"),
    col("DESCRIPTION").alias("tipo_encontro"),
    col("START")
).fillna(0, subset=["custo_total", "cobertura_convenio"]) 

# 2. Preparando a tabela de Condições
df_conditions_opt = df_conditions.select(
    col("PATIENT").alias("patient_id"),
    col("ENCOUNTER").alias("encounter_id"),
    col("DESCRIPTION").alias("diagnostico")
)

# 3. Realizando o JOIN COM BROADCAST (Otimização Explícita)
# O broadcast envia uma cópia da tabela de pacientes para todos os nós,
# evitando o tráfego de rede da tabela grande de encontros (Shuffle).
df_silver_broadcast = df_encounters_opt.join(
    broadcast(df_patients_silver), 
    on="patient_id", 
    how="inner"
).join(
    df_conditions_opt,
    on=["patient_id", "encounter_id"], 
    how="left" 
)

display(df_silver_broadcast.limit(5))

#### Persistindo a Tabela Silver (Delta Lake)

In [ ]:
# Salvando a tabela Silver
# .mode("overwrite") substitui a tabela se ela já existir
# .option("mergeSchema", "true") permite evoluir o schema se adicionarmos colunas no futuro

df_full_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.synthea_data.silver_analitico_saude")

print("Tabela Silver criada com sucesso!")

#### Análise de Negócios - Camada Gold

In [ ]:
from pyspark.sql.functions import sum, count, avg, round, desc

# Agregação: Custo por Diagnóstico e Estado
df_gold_custos = spark.read.table("workspace.synthea_data.silver_analitico_saude") \
    .filter(col("diagnostico").isNotNull()) \
    .groupBy("estado", "diagnostico") \
    .agg(
        sum("custo_total").alias("custo_total_acumulado"),
        round(avg("custo_total"), 2).alias("ticket_medio"),
        count("encounter_id").alias("qtd_casos"),
        round(avg("idade"), 0).alias("idade_media_pacientes")
    ) \
    .orderBy(col("custo_total_acumulado").desc())

display(df_gold_custos)

#### Visualização e Otimização

In [ ]:
# Query SQL direta na tabela Gold (para mostrar interoperabilidade)
# Suponha que salvamos a gold antes (ou usamos TempView)
df_gold_custos.createOrReplaceTempView("view_gold_custos")

spark.sql("""
    SELECT 
        estado, 
        diagnostico, 
        custo_total_acumulado 
    FROM view_gold_custos 
    WHERE qtd_casos > 10 -- Filtrando outliers
    LIMIT 20
""").display()

In [ ]:
df_gold_custos.explain()